# 01d · 정상분포: 세 가지 계산법과 존재·유일성 (Stationary Distributions: Three Ways to Compute, Existence and Uniqueness)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §1.4 (정상분포, social mobility 예제 $\pi=(22,16,9)/47$), §1.6 (Ehrenfest, 이중확률행렬), §1.7 (존재·유일성 증명) · Ross 12e §4.4 (장기 비율, $\pi_j=1/m_{jj}$) · Norris §1.7 (불변측도의 유일성, Thm 1.7.5–1.7.7) · Lawler 2e §1.2 (왼쪽 고유벡터 관점) |
| 선수 노트북 | 01c |
| 예상 소요 | 90분 |
| 상태 | draft |

01a에서 날씨 사슬의 $P^n$은 두 행이 같은 벡터 $(4/7,3/7)$로 수렴했고, 01b–01c에서는 사슬이 **어디에 갇히는가**를 다뤘다. 이 노트북은 갇힌 곳(기약 유한 사슬) 안에서 **시간을 어떻게 나눠 쓰는가**를 묻는다. 답은 정상분포 $\pi$ 하나이고, 그것을 세 가지 방식(선형계·고유벡터·장기 빈도)으로 계산해 같은 벡터임을 확인한다. 그리고 "정상분포가 있다"와 "$P^n$이 그리로 수렴한다"가 **다른 명제**임을 주기적 사슬로 눈에 담는다.

## 0. 준비 (Setup)

In [ ]:
import os
from math import comb
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import matrix_power
from scipy.stats import binom
from IPython.display import Markdown
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.markov import (is_stochastic, simulate_chain, empirical_distribution,
                          stationary, stationary_eig, stationary_power,
                          classify_states, hitting_times, tv_distance, ehrenfest_matrix)
from spkit.plots import plot_hist_vs_pmf

SEED, rng = rng_for("01d")
setup_plots()
N_PATHS = scale(20_000)     # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
N_LONG = scale(200_000)     # 한 경로의 길이 (시간 평균용)
N_BATCHES = 50              # 배치 평균 SE 에 쓰는 블록 수
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}, N_LONG={N_LONG}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (01c) 기본행렬 $N=(I-Q)^{-1}$의 원소 $N_{ij}$의 확률적 의미는?

<details><summary>정답</summary>

일시 상태 $i$에서 출발해 흡수 전에 일시 상태 $j$를 방문하는 횟수(출발 포함)의 기댓값. $N=\sum_{k\ge0}Q^k$의 $(i,j)$ 원소가 $\sum_k P_i(X_k=j)$이기 때문이다.

</details>

**Q2.** (01c) 흡수확률 행렬과 기대 흡수시간을 $N$, $R$로 쓰면?

<details><summary>정답</summary>

$B=NR$, $t=N\mathbf 1$.

</details>

**Q3.** (01c) 도박꾼 파산 $p\ne1/2$에서 $P_i(\text{reach }N)$ 공식과 $p=1/2$에서 $E_i[T]$는?

<details><summary>정답</summary>

$(1-r^i)/(1-r^N)$, $r=q/p$; $p=1/2$이면 $E_i[T]=i(N-i)$.

</details>

**Q4.** (01b) 일시 상태 $x$의 기대 방문 횟수와 귀환확률의 관계는?

<details><summary>정답</summary>

$E_xV_x=1/(1-\rho_{xx})=\sum_{n\ge0}p^n(x,x)$. 재귀 상태에서는 $\rho_{xx}=1$이라 무한대 — 오늘은 재귀 상태에서 '얼마나 자주' 돌아오는지를 $E_xT_x$로 잰다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **정상분포 (stationary distribution)**: 확률벡터 $\pi$ ($\pi\ge0$, $\sum_x\pi(x)=1$)로서
  $$\pi P=\pi,\qquad\text{즉}\qquad \sum_x\pi(x)\,p(x,y)=\pi(y)\quad\forall y.$$
  초기분포가 $\pi$이면 $X_1\sim\pi P=\pi$, 귀납적으로 모든 $n$에서 $X_n\sim\pi$ — 분포가 시간에 대해 **정상(stationary)** 이다. $\pi$는 **행벡터**이고 $P$의 **왼쪽**에서 곱한다.
- **계산법 (i) 선형계**: $\pi(P-I)=0$은 $|S|$개의 방정식이지만, $P-I$의 행합이 0이므로 $|S|$개의 방정식(열)을 모두 더하면 $0=0$ — 하나가 중복이다. 그 하나를 $\sum\pi=1$로 바꾸면 기약 사슬에서 정칙인 선형계가 되어 `np.linalg.solve`로 푼다 — `stationary(P)`.
- **계산법 (ii) 고유벡터**: $\pi P=\pi\iff P^\top\pi^\top=\pi^\top$. 즉 $\pi$는 $P^\top$의 고유값 1에 대한 고유벡터를 합이 1이 되게 정규화한 것(부동소수점의 허수부는 버림) — `stationary_eig(P)`.
- **계산법 (iii) 장기 빈도 / 거듭제곱**: 기약 **비주기**이면 $P^n$의 임의의 행 $\to\pi$ — `stationary_power(P, n)`. 주기가 있어도 한 경로의 **시간 평균** $\frac1n\sum_{k=1}^n\mathbf 1\{X_k=y\}\to\pi(y)$ (에르고딕 정리, 정리 4; 증명은 04c).
- **평균 귀환시간 (mean return time)** $m_x=E_x[T_x]$, $T_x=\min\{n\ge1:X_n=x\}$. 기약 유한 사슬에서 $\pi(x)=1/m_x$ (정리 1).
- **이중확률행렬 (doubly stochastic)**: 행합과 열합이 모두 1인 행렬.
- **Ehrenfest 사슬**: 공 $n$개가 두 항아리에 나뉘어 있고 매 걸음 공 하나를 균등하게 골라 다른 항아리로 옮긴다. 상태 $k$ = 항아리 A의 공 수, $p(k,k+1)=(n-k)/n$, $p(k,k-1)=k/n$ — `ehrenfest_matrix(n)`.
- **반사 랜덤 워크 (reflecting random walk)** on $\{0,\dots,N\}$: 안쪽에서 $\pm1$ 확률 $1/2$, $p(0,1)=p(N,N-1)=1$. 정상분포 $\pi\propto(1,2,2,\dots,2,1)$ (E2), 주기 2 (01b).

### 2.2 정리 1 — 존재와 유일성 (기약 유한 사슬) (Durrett 3e §1.4, §1.7; Norris §1.7 Thm 1.7.5–1.7.7)

> 유한 기약 사슬은 정상분포를 **정확히 하나** 가지며, 모든 $x$에서 $\pi(x)>0$이고
> $$\pi(x)=\frac1{E_x[T_x]}.$$

**가정이 왜 필요한가**
- **유한**: 무한 사슬에서는 기약이어도 정상분포가 없을 수 있다. $\mathbb Z$ 위의 대칭 랜덤 워크는 균등 '측도' $\mu\equiv1$이 $\mu P=\mu$를 만족하지만 $\sum_x\mu(x)=\infty$라 정규화가 안 된다 — $E_0T_0=\infty$(영재귀, 01x)라서 $1/E_xT_x=0$이 되는 것과 같은 말이다.
- **기약**: 닫힌 클래스가 둘이면 각 클래스의 정상분포 $\pi_A,\pi_B$의 볼록결합 $\alpha\pi_A+(1-\alpha)\pi_B$가 전부 정상분포라 유일하지 않다(E3). 일시 상태에는 $\pi(x)=0$이 강제된다(정상분포 아래서 $P(X_n=x)$는 $n$에 무관한데, 일시 상태의 방문은 유한 번뿐이므로 $\sum_n P_\pi(X_n=x)<\infty$).

**증명 스케치.**
*존재.* 기준 상태 $x$를 잡고 **한 사이클** 동안의 방문 횟수
$$\mu(y)=E_x\Big[\sum_{n=0}^{T_x-1}\mathbf 1\{X_n=y\}\Big]$$
를 정의한다. 유한 기약 사슬은 재귀이고 $E_xT_x<\infty$(유한 닫힌 클래스 안에서 $x$에 도달하기 전에 지나는 걸음 수의 기하적 꼬리)이므로 $\sum_y\mu(y)=E_xT_x<\infty$. $\mu P=\mu$인 이유: 사이클 $\{0,\dots,T_x-1\}$의 방문을 한 칸 밀어 $\{1,\dots,T_x\}$로 세어도(시각 $0$과 시각 $T_x$는 둘 다 $x$이므로) 같은 값이고, 밀린 합은 $\sum_z\mu(z)p(z,y)$이다. 따라서 $\pi=\mu/E_xT_x$가 정상분포이며, 사이클 안에서 $x$는 정확히 한 번(시각 0) 방문되므로 $\mu(x)=1$, 즉 $\pi(x)=1/E_xT_x$.
*양수.* 기약이면 $x$에서 $y$로 가는 경로가 있으므로 한 사이클 동안 $y$를 방문할 확률이 양수, $\mu(y)>0$.
*유일.* $\nu P=\nu$인 다른 확률벡터 $\nu$가 있다고 하자. $\nu(y)=\sum_z\nu(z)p(z,y)$를 반복 전개해 $x$를 처음 지나는 시각으로 경로를 나누면 $\nu(y)\ge\nu(x)\mu(y)$를 얻고(Norris Thm 1.7.5), 양변을 합하면 $1\ge\nu(x)E_xT_x$, 즉 $\nu(x)\le\pi(x)$가 모든 $x$에서 성립한다. 두 확률벡터가 성분마다 $\le$이면 같다. $\square$ — 완전한 증명(재생-보상 정리)은 04c에서.

### 2.3 정리 2 — Ehrenfest 정상분포 (Durrett 3e §1.6)

> 공 $n$개의 Ehrenfest 사슬의 정상분포는 $\pi(k)=\binom nk2^{-n}$, 즉 $\text{Bin}(n,1/2)$이다.

**가정이 왜 필요한가**
- 매 걸음 공 **하나**를 **균등하게** 고른다: 이 대칭성이 $\pi(k)p(k,k+1)=\pi(k+1)p(k+1,k)$(세부균형, 01f)를 만든다. 예컨대 항아리 A쪽 공을 더 자주 고르면 이항분포가 아니다.

<details><summary>증명</summary>

**1단계 (이웃 사이의 흐름이 같다).** $0\le k<n$에 대해
$$\pi(k)\,p(k,k+1)=\binom nk2^{-n}\frac{n-k}n,\qquad \pi(k+1)\,p(k+1,k)=\binom n{k+1}2^{-n}\frac{k+1}n.$$
$\binom nk(n-k)=\dfrac{n!}{k!\,(n-k-1)!}=\binom n{k+1}(k+1)$이므로 두 값이 같다.

**2단계 ($\pi P=\pi$).** $p(x,k)>0$인 $x$는 $k\pm1$뿐이므로
$$\sum_x\pi(x)p(x,k)=\pi(k-1)p(k-1,k)+\pi(k+1)p(k+1,k)=\pi(k)p(k,k-1)+\pi(k)p(k,k+1)=\pi(k),$$
두 번째 등호는 1단계를 $(k-1,k)$와 $(k,k+1)$ 쌍에 적용한 것, 마지막 등호는 행합 $p(k,k-1)+p(k,k+1)=1$. 끝점 $k=0$, $k=n$에서는 존재하지 않는 항($\pi(-1)$, $\pi(n+1)$)이 0이고 $p(0,-1)=p(n,n+1)=0$이므로 같은 식이 그대로 성립한다. $\sum_k\binom nk2^{-n}=1$이므로 $\pi$는 확률벡터다. Ehrenfest 사슬은 기약 유한이므로 정리 1에 의해 이것이 유일한 정상분포다. $\square$

</details>

### 2.4 정리 3 — 이중확률행렬의 정상분포 (Durrett 3e §1.6)

> $P$가 이중확률행렬이면 균등분포 $\pi(x)=1/|S|$가 정상분포다(기약이면 유일).

**가정이 왜 필요한가**
- **열합 1**이 전부다: $\sum_x\frac1{|S|}p(x,y)=\frac1{|S|}\sum_xp(x,y)=\frac1{|S|}$.

<details><summary>증명</summary>

위 한 줄이 증명이다. 확률벡터임은 자명하고, 기약이면 정리 1로 유일. 예: 01b의 순환 클래스(순열행렬), 원(cycle) 위의 대칭 랜덤 워크 — 4.4 끝에서 코드로 확인. 대칭 행렬 $P=P^\top$는 항상 이중확률이다. $\square$

</details>

### 2.5 정리 4 — 에르고딕 정리 (시간 평균) (Ross 12e §4.4 장기 비율; Norris §1.7 $\pi_i=1/m_i$; 증명은 04c)

> 기약 유한 사슬에서 임의의 초기상태에 대해
> $$\frac1n\sum_{k=1}^nf(X_k)\ \longrightarrow\ \sum_x\pi(x)f(x)\quad\text{a.s.}$$
> 특히 $f=\mathbf 1_{\{y\}}$를 넣으면 상태 $y$의 방문 빈도 $\to\pi(y)$. **비주기성은 필요 없다.**

**가정이 왜 필요한가**
- **기약**: 시작 클래스에 따라 극한이 달라지지 않게. 닫힌 클래스가 둘이면 시간 평균은 출발한 클래스의 정상분포로 간다(E3).
- **비주기 불필요**: 시간 평균은 진동을 평균 내므로 주기적 사슬(반사 걷기)에서도 성립한다. 반면 $P^n$ 자체의 수렴(01e)에는 비주기성이 필요하다 — 이 차이가 4.4의 실험이다.

증명은 04c에서 재생-보상 정리로: $x$로의 귀환 시각들이 경로를 i.i.d. 사이클로 자르고, 사이클당 $y$ 방문 횟수의 평균이 $\mu(y)$, 사이클 길이의 평균이 $E_xT_x$이므로 큰 수의 법칙으로 빈도 $\to\mu(y)/E_xT_x=\pi(y)$.

### 2.6 직관

$\pi$는 **질량 보존의 평형**이다: 각 상태 $y$로 들어오는 확률 흐름 $\sum_x\pi(x)p(x,y)$가 나가는 흐름 $\pi(y)\cdot1$과 같다. 그래서 $\pi$는 (a) 오래 돌린 뒤 찍은 **스냅사진**의 분포이자, (b) 한 경로가 각 상태에 쓰는 **시간의 비율**이고, (c) $\pi(x)=1/E_xT_x$ — 평균 5.7걸음마다 돌아오는 상태에는 시간의 $1/5.7$을 쓴다. 세 계산법은 같은 벡터를 세 각도에서 본 것이다.

단, (a)의 스냅사진 해석에는 **비주기성**이 필요하다. 반사 걷기는 짝수 시각에 짝수 상태에만 있으므로 $P^{1000}$의 0번 행은 $\pi$가 아니라 $(1/4,0,1/2,0,1/4)$, $P^{1001}$의 0번 행은 $(0,1/2,0,1/2,0)$이다. 그래도 (b)의 시간 평균은 두 스냅사진을 번갈아 평균 내므로 $\pi=(1,2,2,2,1)/8$로 간다.

### 2.7 함정(traps)

1. `stationary_power(P, n)`은 **비주기 기약**일 때만 $\pi$를 준다. 주기적 사슬(반사 걷기, Ehrenfest)에서는 $P^n$의 행이 $\pi$가 아닌 값들 사이를 진동한다 — 정상분포의 존재와 $P^n$의 수렴은 다른 문제.
2. 고유벡터 방법은 고유값 1이 중복(기약 아님)이면 고유공간의 **임의의** 벡터 하나를 돌려주고, 부동소수점 때문에 작은 허수부가 생기므로 `.real`과 정규화가 필요하다.
3. $\pi P=\pi$에서 $\pi$는 행벡터(왼쪽 곱). `np.linalg.eig(P)`는 **오른쪽** 고유벡터를 주므로 $P^\top$에 적용해야 한다. ($P$의 오른쪽 고유벡터는 상수벡터 $\mathbf 1$ — 재미없다.)
4. '정상분포가 있다'와 '어디서 출발하든 그 분포로 수렴한다'는 다르다. 후자는 01e의 수렴 정리(기약 + 비주기).
5. 한 경로의 시간 평균으로 $\pi$를 추정할 때 표본 $\mathbf 1\{X_k=y\}$들은 **독립이 아니다**. 단순 `std/sqrt(n)`은 SE를 과소평가하므로 **배치 평균(batch means)** — 경로를 긴 블록으로 잘라 블록 평균들의 표준편차를 쓴다 — 으로 SE를 구한다.

✍️ **내 말로**: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요) — (1) 정상분포의 정의(흐름 보존으로), (2) 유일성에 왜 기약성이 필요한지, (3) $\pi(x)=1/E_xT_x$의 직관을 각각 두 문장으로.

## 3. Predict — 코드를 돌리기 전에 예측

아래 셀의 `None`을 숫자로 바꿔 보세요. 사회 이동 사슬(Durrett 3e §1.4; 0=상류, 1=중류, 2=하류)은
$$P=\begin{pmatrix}0.7&0.2&0.1\\0.3&0.5&0.2\\0.2&0.4&0.4\end{pmatrix}$$
이고, 첫 항목은 $\pi P=\pi$를 손으로 풀 수 있습니다(E1). Ehrenfest와 날씨 사슬은 이론의 공식으로, 마지막 둘은 2.6의 직관으로 맞힐 수 있습니다. 5절의 비교표에서 이 값이 '내 예측' 열에 들어갑니다.

In [ ]:
predictions = {
    # 사회 이동 사슬 P (위)의 정상분포에서 pi(0) (상류층의 장기 비율)은?
    "pi_mobility_0": None,
    # 공 20개 Ehrenfest 사슬에서 pi(10) (항아리 A에 정확히 절반)은?  힌트: Bin(20, 1/2)
    "pi_ehrenfest_10": None,
    # 상태 10에서 출발해 10으로 처음 돌아오는 평균 시간 E_10[T_10]은?
    "mean_return_ehrenfest_10": None,
    # 01a 날씨 사슬 P = [[0.7, 0.3], [0.4, 0.6]] (0=맑음, 1=비)에서 비 오는 날의 장기 비율 pi(1)은?
    "pi_rain": None,
    # 같은 사슬에서 비 온 날 다음에 다시 비 오기까지의 평균 시간 E_1[T_1]은?
    "mean_return_rain": None,
    # 반사 랜덤 워크 {0,...,4} (p(0,1)=p(4,3)=1, 안쪽 1/2)의 pi(0)은?
    "pi_reflect_0": None,
    # 같은 사슬에서 matrix_power(P, 1000)[0, 0]은?
    "power_reflect_00": None,
}

## 4. Simulate — 시뮬레이션

### 4.1 사회 이동 사슬: 세 계산법이 같은 벡터를 준다

먼저 시뮬레이션 없이 행렬만으로. `stationary`(선형계), `stationary_eig`(고유벡터), `stationary_power`(거듭제곱)를 비교하고, Durrett의 손 계산 $(22,16,9)/47$과 같은지, 그리고 정의 $\pi P=\pi$를 만족하는지 본다. 고유값도 찍어 둔다: 두 번째 고유값 $\lambda_2\approx0.44$가 $P^n$의 수렴 속도(01e)를 정한다.

In [ ]:
P = np.array([[0.7, 0.2, 0.1],
              [0.3, 0.5, 0.2],
              [0.2, 0.4, 0.4]])            # 0 = upper, 1 = middle, 2 = lower
assert is_stochastic(P)

pi_solve = stationary(P)
pi_eig = stationary_eig(P)
pi_pow = stationary_power(P, 100)
pi_hand = np.array([22, 16, 9]) / 47       # Durrett 3e §1.4 의 손 계산
print("linear system :", pi_solve)
print("eigenvector   :", pi_eig)
print("P^100 row 0   :", pi_pow)
print("pi * 47       :", pi_solve * 47)
print("eigenvalues   :", np.sort(np.linalg.eigvals(P).real)[::-1])

assert np.allclose(pi_solve, pi_hand) and np.allclose(pi_eig, pi_solve) and np.allclose(pi_pow, pi_solve)
assert np.allclose(pi_solve @ P, pi_solve), "pi P = pi 가 아니다"
pi_mob = pi_solve

세 벡터가 소수점 15자리까지 같다. `pi_solve @ P`가 다시 `pi_solve`인 것이 정의 그 자체다. 이제 $P^n$의 세 행이 어떻게 하나로 모이는지 그린다 — 세 출발 계층이 $n\approx10$이면 상류층 비율 $22/47$을 잊는다는 뜻이다.

In [ ]:
n_arr = np.arange(0, 16)
rows_n = np.array([matrix_power(P, n)[:, 0] for n in n_arr])     # shape (16, 3): (P^n)[x, 0]

fig, ax = plt.subplots()
for x, lab in enumerate(["from 0 (upper)", "from 1 (middle)", "from 2 (lower)"]):
    ax.plot(n_arr, rows_n[:, x], "o-", ms=4, label=lab)
ax.axhline(pi_mob[0], color="k", ls="--", lw=1, label="pi(0) = 22/47")
ax.set_xlabel("n"); ax.set_ylabel("$(P^n)_{x,0}$")
ax.set_title("Social mobility chain: rows of P^n converge to pi(0) = 22/47")
ax.legend(); plt.show()

### 4.2 스냅사진 vs 시간 평균 — 두 가지 몬테카를로

같은 $\pi(0)$을 두 방식으로 추정한다.

- **스냅사진(독립 경로)**: 상태 2(하류)에서 출발한 경로 `N_PATHS`개를 40걸음 돌리고 시각 40의 단면에서 상태 0의 비율을 센다. $|\lambda_2|^{40}=0.44^{40}\approx6\times10^{-15}$이므로 $P^{40}$의 행은 $\pi$와 기계 정밀도로 같다 — 표본은 독립이므로 `mc_proportion`의 SE가 그대로 맞다.
- **시간 평균(한 경로)**: 경로 **하나**를 `N_LONG`걸음 돌리고 상태 0에 있었던 시간의 비율을 센다(에르고딕 정리). 표본이 상관되어 있으므로 SE는 배치 평균으로 구한다(함정 5).

spkit에 없는 작은 헬퍼 셋을 먼저 정의한다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def batch_means_se(x, n_batches=50):
    """상관된 1-D 시계열 x 의 시간 평균에 대한 SE: 연속 블록 n_batches 개의 평균들의 std/sqrt(n_batches)."""
    x = np.asarray(x, dtype=float)
    m = (x.size // n_batches) * n_batches            # 블록 크기가 같도록 꼬리를 버린다
    block_means = x[:m].reshape(n_batches, -1).mean(axis=1)
    return float(block_means.std(ddof=1) / np.sqrt(n_batches))

def running_frequency(path, state):
    """누적 빈도 (1/n) sum_{k<=n} 1{X_k = state}, n = 1..len(path)."""
    return np.cumsum(path == state) / np.arange(1, len(path) + 1)

def reflecting_walk_matrix(N):
    """01b 와 같은 반사 랜덤 워크 on {0..N}: p(0,1) = p(N,N-1) = 1, 안쪽은 1/2 씩."""
    P_ref = np.zeros((N + 1, N + 1))
    P_ref[0, 1] = P_ref[N, N - 1] = 1.0
    i = np.arange(1, N)
    P_ref[i, i - 1] = P_ref[i, i + 1] = 0.5
    return P_ref

In [ ]:
# (a) 스냅사진: 독립 경로 N_PATHS 개, 시각 40 의 단면
N_SNAP = 40
paths_mob = simulate_chain(P, 2, N_SNAP, rng, n_paths=N_PATHS)      # 모두 하류(2)에서 출발
est_snap = mc_proportion(paths_mob[:, N_SNAP] == 0)
print("P^40 row 2 vs pi  :", np.abs(matrix_power(P, N_SNAP)[2] - pi_mob).max())
print("snapshot  pi(0)   :", est_snap, "   exact", pi_mob[0])

# (b) 시간 평균: 경로 하나, burn-in 100
BURN_MOB = 100
long_mob = simulate_chain(P, 2, N_LONG, rng)[0]
freq_mob = empirical_distribution(long_mob, 3, burn_in=BURN_MOB)
se_mob = batch_means_se(long_mob[BURN_MOB:] == 0, N_BATCHES)
se_naive = np.std(long_mob[BURN_MOB:] == 0, ddof=1) / np.sqrt(N_LONG - BURN_MOB)
print(f"time avg  pi      : {freq_mob}  (exact {pi_mob})")
print(f"pi(0) SE: batch means {se_mob:.4f}  vs naive iid {se_naive:.4f}")

두 SE의 비율이 약 $1.6$이다: 이웃한 시각의 $\mathbf 1\{X_k=0\}$끼리 양의 상관($\rho_1\approx0.44$)이 남아 있어 순진한 i.i.d. SE는 참 SE를 약 40% **과소평가**한다(함정 5). 이 비율은 사슬의 $\lambda_2$만으로 정해지지 않는다 — 4.3에서 같은 사슬의 두 통계량이 전혀 다른 비율을 내는 것을 보고, 기본행렬로 이론값을 계산한다. 아래 그림은 한 경로의 누적 빈도가 $n$이 커지며 $22/47$에 붙는 모습이다.

In [ ]:
run_freq = running_frequency(long_mob, 0)
n_axis = np.arange(1, N_LONG + 2)

fig, ax = plt.subplots()
ax.semilogx(n_axis[9:], run_freq[9:], lw=1, label="running frequency of state 0 (one path)")
ax.axhline(pi_mob[0], color="k", ls="--", lw=1, label="pi(0) = 22/47")
ax.axhspan(freq_mob[0] - 2 * se_mob, freq_mob[0] + 2 * se_mob, xmin=0.85, color="C1", alpha=0.3,
           label="time average ± 2 SE (batch means)")
ax.set_xlabel("n (log scale)"); ax.set_ylabel("fraction of time in state 0")
ax.set_title("Ergodic theorem: time average -> pi(0)")
ax.legend(loc="lower right"); plt.show()

### 4.3 Ehrenfest(20): $\pi=\text{Bin}(20,1/2)$와 $E_{10}T_{10}=1/\pi(10)$

정리 2를 코드로 확인한 뒤, (i) 상태 10으로의 귀환시간 평균이 $1/\pi(10)=2^{20}/\binom{20}{10}$인지(정리 1), (ii) 한 경로의 시간 사용 히스토그램이 이항 pmf와 겹치는지 본다. 귀환시간은 독립 경로 `N_PATHS`개에서 재고, 지평 200은 충분히 길다: $P_{10}(T_{10}>200)\approx3\times10^{-10}$ (금기 행렬 거듭제곱으로 계산할 수 있다 — 아래 셀에서 실제로 확인).

In [ ]:
N_BALLS = 20
E = ehrenfest_matrix(N_BALLS)
pi_E = stationary(E)
pmf_E = binom.pmf(np.arange(N_BALLS + 1), N_BALLS, 0.5)
assert np.allclose(pi_E, pmf_E), "Ehrenfest 정상분포가 Bin(n, 1/2) 가 아니다"
print("period of Ehrenfest(20):", classify_states(E)["period"][0])
print(f"pi(10) = {pi_E[10]:.6f} = C(20,10)/2^20 = {comb(20, 10) / 2**20:.6f},  1/pi(10) = {1 / pi_E[10]:.6f}")

# 지평 절단 확인: 10 을 피하는 '금기(taboo)' 전이로 P_10(T_10 > k) 를 계산
E_taboo = E.copy(); E_taboo[:, 10] = 0.0
v = E[10].copy(); v[10] = 0.0                   # 1 걸음 뒤 10 이 아닌 곳에 있을 확률 벡터
surv = [1.0, v.sum()]
for _ in range(2, 201):
    v = v @ E_taboo; surv.append(v.sum())
surv = np.array(surv)
print(f"P_10(T_10 > 100) = {surv[100]:.2e},  P_10(T_10 > 200) = {surv[200]:.2e}")

In [ ]:
# (i) 상태 10 으로의 귀환시간: 독립 경로 N_PATHS 개, 지평 200
paths_E = simulate_chain(E, 10, 200, rng, n_paths=N_PATHS)
T_E = hitting_times(paths_E[:, 1:], 10) + 1      # T_x 는 n >= 1 부터 (01b 함정 4)
assert np.isfinite(T_E).all(), "지평 200 안에 돌아오지 않은 경로가 있다"
est_T_E = mc_estimate(T_E)
print(f"E_10[T_10]: MC {est_T_E}   exact 1/pi(10) = {1 / pi_E[10]:.4f}")
print("return times are all even:", bool(np.all(T_E % 2 == 0)))

# (ii) 한 경로의 시간 평균 (상태 0 = 빈 항아리에서 출발, burn-in 200): 통계량 1{X=10} 과 X 자체
BURN_E = 200
long_E = simulate_chain(E, 0, N_LONG, rng)[0]
tail_E = long_E[BURN_E:]
freq_E = empirical_distribution(long_E, N_BALLS + 1, burn_in=BURN_E)
se_E = batch_means_se(tail_E == 10, N_BATCHES)
se_E_naive = np.std(tail_E == 10, ddof=1) / np.sqrt(tail_E.size)
mean_X, se_X = tail_E.mean(), batch_means_se(tail_E, N_BATCHES)
se_X_naive = np.std(tail_E, ddof=1) / np.sqrt(tail_E.size)
print(f"time avg pi(10) = {freq_E[10]:.4f} ± {se_E:.4f}  (naive iid SE {se_E_naive:.4f}, ratio {se_E / se_E_naive:.2f})   exact {pi_E[10]:.4f}")
print(f"time avg of X   = {mean_X:.3f} ± {se_X:.3f}  (naive iid SE {se_X_naive:.3f}, ratio {se_X / se_X_naive:.2f})   exact {N_BALLS / 2}")

귀환시간이 전부 짝수인 것은 주기 2 때문이다. 두 시간 평균의 SE 비율은 **통계량에 따라 전혀 다르다**. Ehrenfest는 $\lambda_2=1-2/20=0.9$로 천천히 섞이지만, 지표 $\mathbf 1\{X=10\}$의 배치 평균 SE는 순진한 SE의 약 $1.4$배(이론값 $1.36$)에 그친다 — 4.2의 빨리 섞이는 사슬(이론값 $1.60$)보다 오히려 작다. (블록 50개로 추정한 SE 자체의 상대 오차가 약 10%이므로 관측 비율은 $\pm0.2$쯤 흔들린다.) 이유: $\mathbf 1\{X=10\}$은 중심 $10$에 대해 대칭(우함수)이라 느린 고유값 $0.9$의 고유함수 $k-10$(기함수)에 실리지 않고, 주기 2의 부호 교대($\rho_1<0$, $\rho_2>0$)가 상관을 부분적으로 상쇄한다. 반면 $X$ 자체는 $E[X_{n+1}-10\mid X_n=k]=0.9\,(k-10)$이므로 정확히 $\lambda_2=0.9$의 고유함수이고, 배치 평균 SE가 순진한 SE의 약 $4.4$배다. 즉 함정 5의 크기는 $\lambda_2$가 아니라 **통계량이 느린 고유함수에 얼마나 실리는지**가 정한다. 아래 셀은 이 비율의 이론값($n\to\infty$)을 기본행렬 $Z=(I-P+\mathbf 1\pi)^{-1}$로 계산해 관측치와 맞춰 본다: 점근분산은 $\sigma^2_\infty=\sum_x\pi(x)f_c(x)\,[2(Zf_c)(x)-f_c(x)]$, $f_c=f-\pi f$.

In [ ]:
# helper (spkit에 없어서 여기서 정의): 시간 평균의 점근분산 — 기본행렬 Z = (I - P + 1 pi)^{-1}
def se_ratio_theory(P, f):
    """배치 평균 SE / 순진한 iid SE 의 이론적 비율 sqrt(sigma_inf^2 / Var_pi f)  (n -> inf)."""
    pi = stationary(P); fc = np.asarray(f, float) - pi @ f
    Z = np.linalg.inv(np.eye(len(pi)) - P + np.outer(np.ones(len(pi)), pi))
    return float(np.sqrt(pi @ (fc * (2 * Z @ fc - fc)) / (pi @ fc**2)))

def autocorr(P, f, k):
    """정상 상태에서 corr(f(X_0), f(X_k))."""
    pi = stationary(P); fc = np.asarray(f, float) - pi @ f
    return float(pi @ (fc * (matrix_power(P, k) @ fc)) / (pi @ fc**2))

ks = np.arange(N_BALLS + 1)
cases = [("mobility 1{X=0}", P, np.array([1.0, 0, 0]), se_mob / se_naive),
         ("Ehrenfest 1{X=10}", E, (ks == 10).astype(float), se_E / se_E_naive),
         ("Ehrenfest X", E, ks.astype(float), se_X / se_X_naive)]
for name, M, f, observed in cases:
    print(f"{name:19s} SE ratio: theory {se_ratio_theory(M, f):.2f}, observed {observed:.2f}   "
          f"rho_1 = {autocorr(M, f, 1):+.2f}, rho_2 = {autocorr(M, f, 2):+.2f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
plot_hist_vs_pmf(long_E[BURN_E:], np.arange(N_BALLS + 1), pmf_E, ax=ax1, label_pmf="Bin(20, 1/2) pmf")
ax1.set_xlabel("balls in urn A"); ax1.set_title("One long path: time spent in each state")

k_max = 40
ax2.hist(T_E[T_E <= k_max], bins=np.arange(1.5, k_max + 1.5), density=True, alpha=0.6, label="return times to 10")
ax2.axvline(est_T_E.mean, color="C3", lw=1.5, label=f"MC mean {est_T_E.mean:.3f}")
ax2.axvline(1 / pi_E[10], color="k", ls="--", lw=1, label=f"1/pi(10) = {1 / pi_E[10]:.3f}")
ax2.set_xlabel("k (only even k possible)"); ax2.set_title("Return time to state 10")
ax2.legend()
fig.suptitle("Ehrenfest(20): pi = Bin(20,1/2), E_10[T_10] = 1/pi(10)", y=1.02)
plt.tight_layout(); plt.show()

### 4.4 반사 랜덤 워크: 정상분포는 있지만 $P^n$은 진동한다

$\{0,\dots,4\}$ 위의 반사 걷기는 기약 유한이므로 정상분포 $(1,2,2,2,1)/8$이 유일하게 존재한다(정리 1, 유도는 E2). 그러나 주기 2라서 `stationary_power`는 **틀린 답**을 준다. 세자로(Cesàro) 평균 $\frac1n\sum_{k=1}^nP^k$의 행과 한 경로의 시간 평균은 $\pi$로 간다.

In [ ]:
N_REF = 4
P_ref = reflecting_walk_matrix(N_REF)
pi_ref = stationary(P_ref)
pow_even = stationary_power(P_ref, 1000)           # 첫 행 of P^1000
pow_odd = matrix_power(P_ref, 1001)[0]
cesaro = sum(matrix_power(P_ref, k) for k in range(1, 201)) / 200
print("period            :", classify_states(P_ref)["period"][0])
print("pi (linear system):", pi_ref, " = (1,2,2,2,1)/8:", np.allclose(pi_ref, np.array([1, 2, 2, 2, 1]) / 8))
print("P^1000 row 0      :", pow_even, "  <- NOT pi (the trap)")
print("P^1001 row 0      :", pow_odd)
print("Cesaro(200) row 0 :", cesaro[0], f"  TV to pi = {tv_distance(cesaro[0], pi_ref):.4f}")
assert np.allclose(pow_even, [0.25, 0, 0.5, 0, 0.25]) and np.allclose(pow_odd, [0, 0.5, 0, 0.5, 0])

# 한 경로의 시간 평균 (burn-in 100) + 배치 평균 SE
BURN_R = 100
long_R = simulate_chain(P_ref, 0, N_LONG, rng)[0]
freq_R = empirical_distribution(long_R, N_REF + 1, burn_in=BURN_R)
se_R = batch_means_se(long_R[BURN_R:] == 0, N_BATCHES)
print(f"time avg pi(0) = {freq_R[0]:.4f} ± {se_R:.4f}   exact 1/8")

$P^{1000}$의 행은 짝수 상태에만, $P^{1001}$의 행은 홀수 상태에만 질량을 둔다 — 둘의 평균이 정확히 $\pi$다. 세자로 평균은 200항으로 TV 거리 $0.0075$까지 갔고, 시간 평균은 SE 안에서 $1/8$이다.

In [ ]:
states = np.arange(N_REF + 1)
series = {"pi (exact)": pi_ref, "P^1000 row 0": pow_even, "P^1001 row 0": pow_odd,
          "Cesaro average (200)": cesaro[0], "time average (one path)": freq_R}
w = 0.16
fig, ax = plt.subplots(figsize=(8, 4))
for j, (lab, vals) in enumerate(series.items()):
    ax.bar(states + (j - 2) * w, vals, width=w, label=lab)
ax.set_xticks(states); ax.set_xlabel("state"); ax.set_ylabel("probability")
ax.set_title("Periodic chain: stationary distribution exists, but P^n oscillates")
ax.legend(fontsize=8); plt.show()

마지막으로 정리 3(이중확률 ⇒ 균등)을 두 예로 확인한다: 01b의 순환 클래스(순열행렬, 주기 3)와 원 $\mathbb Z_5$ 위의 대칭 랜덤 워크(길이 2와 5의 귀환이 모두 가능하므로 주기 1). 순열행렬은 주기 3이라 `stationary_power`로는 안 되지만 `stationary`(선형계)는 주기와 무관하게 균등분포를 준다.

In [ ]:
P_cycle3 = np.roll(np.eye(3), 1, axis=1)                         # 0->1->2->0 (permutation matrix)
P_ring5 = 0.5 * (np.roll(np.eye(5), 1, axis=1) + np.roll(np.eye(5), -1, axis=1))   # symmetric walk on Z_5
for name, M in [("cycle permutation (3)", P_cycle3), ("symmetric walk on Z_5", P_ring5)]:
    doubly = is_stochastic(M) and is_stochastic(M.T)
    print(f"{name:24s} doubly stochastic: {doubly}, stationary = {stationary(M)}, period = {classify_states(M)['period'][0]}")

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식은 모두 코드로 계산한다: 사회 이동은 `stationary(P)`(손 계산 $22/47$과 이미 일치 확인), Ehrenfest는 $\binom{20}{10}2^{-20}$과 그 역수, 날씨 사슬은 $\pi=(0.4,0.3)/0.7$과 $1/\pi(1)$, 반사 걷기는 $(1,2,2,2,1)/8$. 날씨 사슬 두 행과 $P^{1000}$ 행은 MC가 아니므로 SE $=0$(정확 비교)이다.

In [ ]:
P_weather = np.array([[0.7, 0.3], [0.4, 0.6]])                     # 01a: 0 = Sunny, 1 = Rainy
a, b = P_weather[0, 1], P_weather[1, 0]
pi_weather = np.array([b, a]) / (a + b)                              # 2-상태 사슬의 닫힌 형식
assert np.allclose(pi_weather, stationary(P_weather))

pi_mob0_exact = float(pi_mob[0])                                     # = 22/47
pi_E10_exact = comb(20, 10) / 2**20                                  # 정리 2
mean_return_E10_exact = 1 / pi_E10_exact                             # 정리 1
pi_ref0_exact = 1 / (2 * N_REF)                                      # E2: pi(0) = 1/(2N)

rows = [
    {"name": "pi_mobility_0 (snapshot, t=40)", "predicted": predictions["pi_mobility_0"], "estimate": est_snap, "exact": pi_mob0_exact},
    {"name": "pi_mobility_0 (time average)", "predicted": predictions["pi_mobility_0"], "estimate": float(freq_mob[0]), "exact": pi_mob0_exact, "se": se_mob},
    {"name": "pi_ehrenfest_10 (time average)", "predicted": predictions["pi_ehrenfest_10"], "estimate": float(freq_E[10]), "exact": pi_E10_exact, "se": se_E},
    {"name": "mean_return_ehrenfest_10", "predicted": predictions["mean_return_ehrenfest_10"], "estimate": est_T_E, "exact": mean_return_E10_exact},
    {"name": "pi_rain (not MC)", "predicted": predictions["pi_rain"], "estimate": float(stationary(P_weather)[1]), "exact": float(pi_weather[1]), "se": 0.0},
    {"name": "mean_return_rain = 1/pi(1) (not MC)", "predicted": predictions["mean_return_rain"], "estimate": float(1 / stationary(P_weather)[1]), "exact": float(1 / pi_weather[1]), "se": 0.0},
    {"name": "pi_reflect_0 (time average)", "predicted": predictions["pi_reflect_0"], "estimate": float(freq_R[0]), "exact": pi_ref0_exact, "se": se_R},
    {"name": "P^1000[0,0] of reflecting walk (trap: != pi(0))", "predicted": predictions["power_reflect_00"], "estimate": float(pow_even[0]), "exact": None},
]
print(f"exact: pi_mob(0)={pi_mob0_exact:.6f}, pi_E(10)={pi_E10_exact:.6f}, 1/pi_E(10)={mean_return_E10_exact:.6f}, "
      f"pi_rain={pi_weather[1]:.6f}, E_1T_1={1 / pi_weather[1]:.6f}, pi_ref(0)={pi_ref0_exact}")
Markdown(compare_table(rows))

행별로 '불일치'가 뜻하는 것:
- **pi_mobility_0 (snapshot)**: 어긋나면 $P^{40}$의 행이 아직 $\pi$가 아니거나(수렴이 느린 사슬이라면 지평을 늘려야) 단면 시각을 잘못 잡은 것.
- **pi_mobility_0 (time average)**: 어긋나면 에르고딕 정리가 의심되기 전에 SE부터 의심 — 배치 수가 너무 적거나 블록이 상관 길이보다 짧은 것(함정 5).
- **pi_ehrenfest_10 (time average)**: 어긋나면 `ehrenfest_matrix`의 전이확률이 $(n-k)/n$, $k/n$이 아니거나 burn-in이 부족한 것(빈 항아리에서 출발했다).
- **mean_return_ehrenfest_10**: 어긋나면 `paths[:, 1:]`로 자르지 않아 $T=0$이 섞였거나(01b 함정 4) 지평 200 안에 돌아오지 못한 경로를 버린 것 — 정리 1의 $\pi(x)=1/E_xT_x$ 자체가 틀릴 수는 없다.
- **pi_rain, mean_return_rain**: SE $=0$의 정확 비교. 예측이 어긋났다면 $\pi=(b,a)/(a+b)$에서 $a,b$의 순서를 바꾼 것($\pi(\text{비})$는 맑음→비 확률 $0.3$에 비례).
- **pi_reflect_0 (time average)**: 어긋나면 주기 2 사슬에서 블록 길이가 홀수라 진동이 덜 평균된 것이거나 `reflecting_walk_matrix`의 경계가 틀린 것.
- **P^1000[0,0]**: 닫힌 형식 칸이 비어 있다 — 이 값은 $\pi(0)=0.125$가 **아니라** $0.25$이며, 그것이 함정 1이다.

In [ ]:
assert_close(est_snap, pi_mob0_exact, k=4, name="pi_mobility_0 snapshot")
assert_close(float(freq_mob[0]), pi_mob0_exact, se=se_mob, k=4, name="pi_mobility_0 time average")
assert_close(float(freq_E[10]), pi_E10_exact, se=se_E, k=4, name="pi_ehrenfest_10")
assert_close(est_T_E, mean_return_E10_exact, k=4, name="mean_return_ehrenfest_10")
assert_close(float(freq_R[0]), pi_ref0_exact, se=se_R, k=4, name="pi_reflect_0")
print("all checks passed")

## 6. 연습문제

### E1 계산

사회 이동 사슬 $P=\begin{pmatrix}0.7&0.2&0.1\\0.3&0.5&0.2\\0.2&0.4&0.4\end{pmatrix}$의 정상분포를 **손으로** 구하시오($\pi P=\pi$의 세 식 중 둘 + 정규화). 상류층 사람(의 후손)이 다시 상류층으로 돌아오는 평균 세대 수 $E_0T_0$은? 다 푼 뒤 `stationary(P)`와 `1/stationary(P)[0]`으로 검산하시오.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$\pi P=\pi$의 첫 두 열:
$$0.7\pi_0+0.3\pi_1+0.2\pi_2=\pi_0\ \Rightarrow\ 0.3\pi_0=0.3\pi_1+0.2\pi_2,\qquad 0.2\pi_0+0.5\pi_1+0.4\pi_2=\pi_1\ \Rightarrow\ 0.5\pi_1=0.2\pi_0+0.4\pi_2.$$
첫 식에서 $\pi_2=\frac32(\pi_0-\pi_1)$. 둘째 식에 넣으면 $0.5\pi_1=0.2\pi_0+0.6\pi_0-0.6\pi_1$, 즉 $1.1\pi_1=0.8\pi_0$, $\pi_1=\frac8{11}\pi_0$. 그러면 $\pi_2=\frac32\cdot\frac3{11}\pi_0=\frac9{22}\pi_0$. 따라서 $\pi\propto(1,\frac8{11},\frac9{22})\propto(22,16,9)$, 합 47:
$$\pi=\Big(\frac{22}{47},\frac{16}{47},\frac9{47}\Big)=(0.4681,\ 0.3404,\ 0.1915).$$
세 번째 식 $0.1\pi_0+0.2\pi_1+0.4\pi_2=\pi_2$는 자동으로 성립한다(중복 방정식): $2.2+3.2+3.6=9$. ✓

$E_0T_0=1/\pi_0=47/22\approx2.136$세대.

검산 코드:
```python
pi_hand = np.array([22, 16, 9]) / 47
print(np.allclose(stationary(P), pi_hand), 1 / stationary(P)[0], 47 / 22)   # True 2.136... 2.136...
```

</details>

### E2 유도 후 시뮬레이션 검증

반사 랜덤 워크 $\{0,\dots,N\}$의 정상분포가 $\pi\propto(1,2,2,\dots,2,1)$임을 $\pi P=\pi$로 유도하시오(또는 $\pi(x)p(x,x+1)=\pi(x+1)p(x+1,x)$를 확인). $N=4$에서 한 경로 `N_LONG`걸음의 시간 평균으로 $\pi(0)=1/8$과 $\pi(2)=1/4$를 배치 평균 SE와 함께 `assert_close`로 확인하시오. `stationary_power`로는 왜 안 되는지도 답하시오.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

**유도.** $y$로 들어오는 흐름을 쓴다. 끝점: $\pi(0)=\frac12\pi(1)$ (0으로 오는 길은 1에서 확률 $1/2$뿐). 끝 옆: $\pi(1)=1\cdot\pi(0)+\frac12\pi(2)$. 안쪽 $2\le y\le N-2$: $\pi(y)=\frac12\pi(y-1)+\frac12\pi(y+1)$.
$\pi(0)=c/2$로 두면 $\pi(1)=c$, 다음 식에서 $\pi(2)=2(\pi(1)-\pi(0))=c$, 안쪽 식은 $\pi(y+1)=2\pi(y)-\pi(y-1)$이므로 귀납적으로 $\pi(y)=c$ ($1\le y\le N-1$), 대칭으로 $\pi(N)=\pi(0)=c/2$. 정규화 $2\cdot\frac c2+(N-1)c=Nc=1$에서 $c=1/N$: $\pi(0)=\pi(N)=\frac1{2N}$, 안쪽은 $\frac1N$. $N=4$: $(1,2,2,2,1)/8$.
(세부균형으로도: $\pi(0)p(0,1)=\frac1{2N}\cdot1=\frac1N\cdot\frac12=\pi(1)p(1,0)$, 안쪽은 $\frac1N\cdot\frac12$로 양쪽이 같다.)

**왜 `stationary_power`는 안 되나.** 주기 2이므로 $P^n$의 행은 짝수/홀수 상태 사이를 번갈아 진동하고 수렴하지 않는다(4.4: $P^{1000}[0,:]=(1/4,0,1/2,0,1/4)$). 정상분포의 존재(정리 1)와 $P^n$의 수렴(01e, 비주기 필요)은 다른 명제이며, 시간 평균은 에르고딕 정리(정리 4)로 주기와 무관하게 수렴한다.

시뮬레이션 코드 (SE $\approx0.0012$ full, $0.004$ FAST):
```python
long_R2 = simulate_chain(P_ref, 2, N_LONG, rng)[0]
freq_R2 = empirical_distribution(long_R2, 5, burn_in=100)
for s, exact in [(0, 1 / 8), (2, 1 / 4)]:
    se_s = batch_means_se(long_R2[100:] == s, N_BATCHES)
    print(f"pi({s}): time avg {freq_R2[s]:.4f} ± {se_s:.4f}  exact {exact}")
    assert_close(float(freq_R2[s]), exact, se=se_s, k=4, name=f"pi_reflect_{s}")
print("stationary_power:", stationary_power(P_ref, 1000), " vs pi:", stationary(P_ref))
```

</details>

### E3 가정을 깨보기

기약성을 깨자:
$$P=\begin{pmatrix}0.5&0.5&0&0\\0.2&0.8&0&0\\0&0&0.9&0.1\\0&0&0.5&0.5\end{pmatrix}$$
(닫힌 클래스 $\{0,1\}$, $\{2,3\}$ 둘). `stationary(P)`, `stationary_eig(P)`, `np.linalg.eigvals(P)`를 실행하고 무슨 일이 생기는지 설명하시오. 상태 0과 상태 2에서 출발한 긴 경로의 시간 평균은 각각 무엇으로 수렴하는가? 정상분포 **전체 집합**을 기술하시오.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

고유값은 $\{1,1,0.3,0.4\}$ — 고유값 1의 중복도가 2(닫힌 클래스 수)다. 그래서
- `stationary(P)`: 방정식 하나를 정규화로 바꿔도 여전히 방정식 하나가 더 중복이라 선형계가 **특이(singular)** — `np.linalg.LinAlgError: Singular matrix`가 난다.
- `stationary_eig(P)`: 고유값 1의 고유공간이 2차원이므로 `eig`가 돌려준 (구현 의존적인) 벡터 하나를 정규화한다 — 여기서는 $(2/7,5/7,0,0)$이 나오지만 이것은 정상분포 '하나'일 뿐이다(함정 2).

클래스별 정상분포는 2-상태 공식 $(b,a)/(a+b)$ ($a$ = 첫 상태에서 나갈 확률, $b$ = 둘째 상태에서 나갈 확률)로 $\pi_A=(0.2,0.5)/0.7=(2/7,5/7)$, $\pi_B=(0.5,0.1)/0.6=(5/6,1/6)$. 정상분포 전체 집합은 볼록결합
$$\{\alpha(\pi_A,0,0)+(1-\alpha)(0,0,\pi_B):\ \alpha\in[0,1]\},$$
즉 선분 하나 — 유일성에 기약이 필요한 이유(정리 1). 시간 평균은 출발한 클래스의 것으로 간다: 0에서 출발하면 $(2/7,5/7,0,0)$, 2에서 출발하면 $(0,0,5/6,1/6)$ — 에르고딕 정리(정리 4)에 '기약'이 필요한 이유.

확인 코드:
```python
P3 = np.array([[0.5, 0.5, 0, 0], [0.2, 0.8, 0, 0], [0, 0, 0.9, 0.1], [0, 0, 0.5, 0.5]])
try:
    print(stationary(P3))
except np.linalg.LinAlgError as e:
    print("stationary(P3) ->", e)                       # Singular matrix
print("eig method:", stationary_eig(P3))                # one point of the segment
print("eigenvalues:", np.round(np.linalg.eigvals(P3).real, 4))   # 1 appears twice
print("class A:", stationary(P3[:2, :2]), " class B:", stationary(P3[2:, 2:]))   # (2/7, 5/7), (5/6, 1/6)
for x0 in (0, 2):
    path = simulate_chain(P3, x0, N_LONG, rng)[0]
    print(f"time average from {x0}:", np.round(empirical_distribution(path, 4, burn_in=100), 3))
```

</details>

## 7. 정리

1. 정상분포 $\pi P=\pi$ = 확률 흐름의 평형(들어오는 흐름 = 나가는 흐름); 초기분포가 $\pi$이면 영원히 $\pi$.
2. 세 계산법 — 선형계(`stationary`), $P^\top$의 고유벡터(`stationary_eig`), 장기 빈도/거듭제곱(`stationary_power`, 시간 평균) — 기약 유한 사슬이면 같은 답. 사회 이동 사슬: $(22,16,9)/47$.
3. 기약 유한 ⇒ 정상분포가 존재·유일·양수이고 $\pi(x)=1/E_xT_x$; 기약이 아니면 닫힌 클래스별 정상분포의 볼록결합만큼 많다(E3).
4. Ehrenfest(20): $\pi=\text{Bin}(20,1/2)$, $E_{10}T_{10}=2^{20}/\binom{20}{10}=5.675$; 이중확률행렬 ⇒ 균등; 반사 걷기 ⇒ $(1,2,\dots,2,1)/2N$ — 모두 시뮬레이션과 4 SE 안에서 일치했다.
5. 정상분포의 존재 ≠ $P^n$의 수렴: 주기적 사슬은 시간 평균(그리고 세자로 평균)만 수렴한다. 한 경로의 시간 평균에는 배치 평균 SE를 쓴다.

**trap 카드**
- Q: 정상분포가 존재하고 유일하면 $P^n$의 행은 반드시 $\pi$로 수렴하는가?
- A: 아니다. 반사 랜덤 워크(주기 2)는 $\pi=(1,2,2,2,1)/8$이 유일하지만 $P^{1000}[0,:]=(1/4,0,1/2,0,1/4)$, $P^{1001}[0,:]=(0,1/2,0,1/2,0)$로 진동한다. 수렴에는 비주기성이 추가로 필요하고(01e), 시간 평균은 주기와 무관하게 $\pi$로 간다.

**다음 노트북: 01e** — $P^n\to\pi$가 **언제**, **얼마나 빨리** 성립하는가? 수렴 정리(기약 + 비주기)와 그 속도 — 오늘 찍어 둔 두 번째 고유값 $\lambda_2$, 전변동 거리, 혼합시간 — 를 다룬다.

log/progress.md 한 줄 템플릿:
`- YYYY-MM-DD 01d 정상분포 | 소요 __분 | 예측 적중 _/7 | 막힌 곳: ___ | 다음: 01e`